In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12")

In [0]:
df.limit(5).display()

In [0]:
df = df.dropDuplicates()

In [0]:
COL_RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name, new_name in COL_RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
for field in df.schema.fields:
    if field.dataType == StringType():
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df = df.withColumn(
    "customer_number",
    F.when(col("customer_number").startswith("NAS"),
           F.substring(col("customer_number"), 4, F.length(("customer_number"))))                                 
           .otherwise(col("customer_number")          
           )
)

In [0]:
df.filter(col("birth_date") > F.current_date()).count()

In [0]:
df = df.withColumn(
    "birth_date",
    F.when(col("birth_date") > F.current_date(),
           None)                          
    .otherwise(col("birth_date"))       
)

In [0]:
df.select("gender").distinct().show()

In [0]:
df = df.withColumn(
    "gender", 
    F.when(col("gender").isin("Male", "M"), "Male")
    .when(col("gender").isin("Female", "F"), "Female")
    .otherwise("unknown")
)

In [0]:
df.limit(5).display()

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.silver.erp_customers")

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customers LIMIT(5)